# VersaMammo segmentation quality versus OOD distance

Load a ZGT-trained EfficientNet-B5 U-Net, compute one OOD feature vector per image,
segment MAMA-MIA and INbreast, and correlate KNN distance to the ZGT ID reference
with Dice. Segmentation is automatic: **no ground-truth boxes or masks are supplied
as model inputs**. Unlike MedSAM's notebook, no prompt localization is provided.

OOD feature space: a separate ViT-B/14 encoder, producing 768-dimensional vectors
with the same 224px preprocessing and pooling as the feature extraction notebooks.
EfficientNet-B5 still performs segmentation. The existing ID features are loaded;
ID feature extraction is disabled by default.

**Weight compatibility:** supply the exact ViT encoder state that generated the saved ID
features. The original notebooks skipped the incompatible EfficientNet checkpoint weights,
so a different pretrained ViT or a newly initialized ViT would not recreate that space.
The saved feature tensors alone cannot recover the original encoder weights.

Run cells top to bottom after setting data/checkpoint paths. Requires the existing
VersaMammo segmentation environment plus `pytorch-ood`, `scikit-learn` (KNN),
`scipy`, `matplotlib`, and Pillow. Model weights and datasets are not bundled here.


All experiment logic is defined and executed in this notebook; no separate pipeline script is required.

In [7]:
MODEL_CODE_RELATIVE_PATH = "VersaMammo/downstream/Segment"
from pathlib import Path
import json
import sys
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "pipelines_and_experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "root_utils").is_dir():
    raise RuntimeError("Start Jupyter in FM_thesis or pipelines_and_experiments, or set PROJECT_ROOT explicitly.")

MODEL_CODE_DIR = PROJECT_ROOT / MODEL_CODE_RELATIVE_PATH
for path in [PROJECT_ROOT, MODEL_CODE_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))


DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda:0


In [8]:
# Update these paths for the machine on which inference will run.
DATASETS_ROOT = Path("/home/jovyan/thesis_project/datasets")
EVALUATION_ROOTS = {
    "mamamia": DATASETS_ROOT / "MamaMIA" / "extracted_slices",
    "inbreast": DATASETS_ROOT / "INbreast" / "INbreast Release 1.0" / "MassCases_cropped",
}
ZGT_DATA_ROOT = Path("/mnt/data/spathak")
# Prefer a CSV restricted to the ID training/reference split; mass rows are selected.
ZGT_METADATA_CSV = ZGT_DATA_ROOT / "CLaM-Annot-metadata.csv"

REEXTRACT_ID_FEATURES = False
ALLOW_LEGACY_ID_FEATURES = True
KNN_K = 50  # k-th nearest ID neighbor; lower this if using fewer than 50 ID images
KNN_NORMALIZE_FEATURES = True  # L2-normalize embeddings before computing distances
MAX_ID_CASES = None
MAX_EVALUATION_CASES_PER_DATASET = None  # e.g. 5 for a smoke run
SAVE_PRED_MASKS = False
IMAGE_NPZ_KEY = None  # None selects a role-specific key or the sole array
MASK_NPZ_KEY = None

# Supply an actual ZGT-trained segmentation checkpoint (full U-Net state dict).
SEGMENTATION_CHECKPOINT = PROJECT_ROOT.parent / "models" / "VersaMammo_ZGT_folds" / "VersaMammo_ZGT_fold_0.pth" 
SEGMENTATION_INPUT_SIZE = 512
# Match the policy used to train/evaluate your checkpoint. Choices:
# percentile (evaluate_mamamia default), minmax, legacy (uint8 / 255, unit floats unchanged).
SEGMENTATION_NORMALIZATION = "percentile"
PERCENTILE_LOW, PERCENTILE_HIGH = 1.0, 99.0
MASK_THRESHOLD = 0.5
OUTPUT_ROOT = PROJECT_ROOT / "results" / "pipelines_and_experiments" / "SEG_OOD_VersaMammo"
ID_FEATURES_PATH = PROJECT_ROOT / "feature_space_inspection" / "VersaMammo_ZGT_allmasses"

# Reuse the existing 768-dimensional ViT ID features.
# Encoder weights, preprocessing and pooling must match the evaluation features.

# Exact saved state of the ViT encoder that generated the ID feature tensor.
# This is a separate checkpoint from the EfficientNet segmentation checkpoint.
# Do NOT point this to VersaMammo (Enb5).pth; incompatible weights are rejected.
VIT_CHECKPOINT = PROJECT_ROOT.parent / "models" / "VersaMammo_pretrained" / "VersaMammo (ViT-B).pth"

## Dataset loading

The pipeline definitions are included in notebook cells. These cells define helpers; the following execution cells run them.

In [9]:
from __future__ import annotations
import csv
import hashlib
import json
import warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F


def checkpoint_identity(path):
    """Include checkpoint path, size and modification time in cache metadata."""
    path = Path(path).expanduser().resolve()
    stat = path.stat()
    return {"path": str(path), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}


def collect_paired_cases(root, dataset, limit=None):
    """Pair image/img + mask, or {case_id}_image/img + {case_id}_mask.

    Supports NPZ, NPY and image formats, including multiple named pairs in one
    directory. Each mask must have the same case prefix and parent directory as
    its image. Relative image paths identify cases uniquely across subfolders.
    Missing or ambiguous pairs are rejected instead of silently choosing one.
    """
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(root)
    extensions = {".npz", ".npy", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}

    def image_prefix(path):
        stem = path.stem.lower()
        if stem in {"image", "img"}:
            return ""
        for suffix in ("_image", "_img", "_images"):
            if stem.endswith(suffix) and len(stem) > len(suffix):
                return stem[:-len(suffix)]
        return None

    images = sorted(p for p in root.rglob("*") if p.is_file()
                    and p.suffix.lower() in extensions
                    and image_prefix(p) is not None
                    and not any(part.startswith(".") for part in p.relative_to(root).parts))
    cases = []
    for image in images:
        prefix = image_prefix(image)
        mask_stem = f"{prefix}_mask" if prefix else "mask"
        siblings = [p for p in image.parent.iterdir() if p.is_file()
                    and p.suffix.lower() in extensions]
        masks = [p for p in siblings if p.stem.lower() == mask_stem]
        image_candidates = [p for p in siblings if image_prefix(p) == prefix]
        if len(masks) != 1 or len(image_candidates) != 1:
            raise ValueError(
                f"Expected exactly one image and matching {mask_stem} file in "
                f"{image.parent}; found {len(image_candidates)} images and {len(masks)} masks"
            )
        cases.append({"dataset": dataset, "case_id": image.relative_to(root).as_posix(),
                      "image_path": str(image), "mask_path": str(masks[0])})
    return validate_cases(cases, limit)


def collect_zgt_cases(root, metadata_csv, limit=None):
    """Read mass images from the semicolon-delimited CLaM metadata CSV.

    Use a CSV containing the intended ID reference split. Multiple annotation
    rows for one image are merged into one case with a union ground-truth mask.
    """
    root = Path(root).expanduser().resolve()
    grouped = {}
    with Path(metadata_csv).open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle, delimiter=";")
        required = {"ImagePath", "ROIPath", "AbnormalityType"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f"ZGT CSV needs columns {sorted(required)}")
        for row in reader:
            if row["AbnormalityType"] != "Mass":
                continue
            image_name = row["ImagePath"].strip()
            mask_name = row["ROIPath"].strip()
            if not image_name or image_name.lower() in {"none", "nan"}:
                raise ValueError("Mass row has no ImagePath")
            case = grouped.setdefault(image_name, {
                "dataset": "zgt_id", "case_id": image_name,
                "image_path": str(root / image_name), "mask_paths": [],
            })
            if mask_name and mask_name.lower() not in {"none", "nan"}:
                mask_path = str(root / mask_name)
                if mask_path not in case["mask_paths"]:
                    case["mask_paths"].append(mask_path)
    return validate_cases([grouped[k] for k in sorted(grouped)], limit)


def validate_cases(cases, limit=None):
    if limit is not None and limit < 1:
        raise ValueError("limit must be positive or None")
    cases = list(cases)
    keys = [(c["dataset"], c["case_id"]) for c in cases]
    if len(keys) != len(set(keys)):
        raise ValueError("Duplicate (dataset, case_id) identifiers")
    cases = cases if limit is None else cases[:limit]
    if not cases:
        raise ValueError("No cases found; check the dataset root/split")
    for case in cases:
        paths = [case["image_path"]] + case.get("mask_paths", [case.get("mask_path")])
        for path in paths:
            if path is not None and not Path(path).is_file():
                raise FileNotFoundError(path)
    return cases


def load_array(path, role="image", npz_key=None):
    path = Path(path)
    if path.suffix.lower() == ".npy":
        return np.load(path, allow_pickle=False)
    if path.suffix.lower() == ".npz":
        with np.load(path, allow_pickle=False) as archive:
            if npz_key is not None:
                return archive[npz_key].copy()
            preferred = ("image", "img", "data", "arr_0") if role == "image" else ("mask", "seg", "label", "arr_0")
            if len(archive.files) == 1:
                return archive[archive.files[0]].copy()
            for key in preferred:
                if key in archive:
                    return archive[key].copy()
            raise ValueError(f"Select an NPZ key for {path}: {archive.files}")
    from PIL import Image
    with Image.open(path) as image:
        return np.asarray(image).copy()


def as_2d(array, role="image"):
    """Convert a grayscale/CHW/HWC slice to HW; reject volumes."""
    array = np.asarray(array)
    if array.ndim == 3:
        if array.shape[-1] in (1, 3, 4):
            array = (array[..., :3].mean(axis=-1).astype(array.dtype)
                     if role == "image" and array.shape[-1] > 1 else array[..., 0])
        elif array.shape[0] in (1, 3, 4):
            array = (array[:3].mean(axis=0).astype(array.dtype)
                     if role == "image" and array.shape[0] > 1 else array[0])
    if array.ndim != 2:
        raise ValueError(f"Expected a 2D {role} slice, got {array.shape}")
    if not np.isfinite(array).all():
        raise ValueError(f"Non-finite values in {role}")
    return array


def read_case(case, image_npz_key=None, mask_npz_key=None):
    image = as_2d(load_array(case["image_path"], "image", image_npz_key))
    paths = case.get("mask_paths", [case.get("mask_path")])
    mask = np.zeros(image.shape, dtype=bool)
    for path in paths:
        if path is None:
            continue
        component = as_2d(load_array(path, "mask", mask_npz_key), "mask")
        if component.shape != image.shape:
            raise ValueError(f"Image/mask shape mismatch for {case['case_id']}")
        mask |= component > 0
    return image, mask

## Feature extraction, segmentation evaluation, and OOD scoring

In [10]:
def binary_metrics(prediction, target):
    prediction, target = np.asarray(prediction, dtype=bool), np.asarray(target, dtype=bool)
    if prediction.shape != target.shape or prediction.ndim != 2:
        raise ValueError(f"Expected matching HW masks: {prediction.shape}, {target.shape}")
    pred_area, gt_area = int(prediction.sum()), int(target.sum())
    intersection = int((prediction & target).sum())
    union = pred_area + gt_area - intersection
    return {"dice": 2 * intersection / (pred_area + gt_area) if pred_area + gt_area else 1.0,
            "iou": intersection / union if union else 1.0,
            "gt_area": gt_area, "pred_area": pred_area,
            "gt_fraction": gt_area / target.size,
            "evaluation_height": target.shape[0], "evaluation_width": target.shape[1]}


def extract_reference_features(cases, feature_fn, output_path, signature, **read_kwargs):
    vectors = []
    for index, case in enumerate(cases, 1):
        image, _ = read_case(case, **read_kwargs)
        vectors.append(feature_fn(image).detach().cpu().reshape(1, -1))
        if index % 10 == 0 or index == len(cases):
            print(f"ID features: {index}/{len(cases)}")
    features = torch.cat(vectors)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"features": features, "signature": signature, "cases": cases}, output_path)
    return features


def load_reference_features(path, signature, allow_legacy=False):
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if torch.is_tensor(payload):
        if not allow_legacy:
            raise ValueError("Legacy ID tensor has no encoder/preprocessing metadata. Re-extract it "
                             "or set ALLOW_LEGACY_ID_FEATURES=True after checking its provenance.")
        warnings.warn("Using legacy ID features: verify the same encoder weights, fold and preprocessing")
        return payload.float()
    if payload["signature"] != signature:
        raise ValueError("ID feature encoder/preprocessing signature differs. Re-extract ID features.")
    return payload["features"].float()


def evaluate_cases(cases, feature_fn, predict_fn, output_dir, save_masks=False, **read_kwargs):
    """predict_fn(image) returns (binary prediction, evaluation HW)."""
    rows, vectors = [], []
    output_dir = Path(output_dir)
    for index, case in enumerate(cases, 1):
        image, original_gt = read_case(case, **read_kwargs)
        vector = feature_fn(image).detach().cpu().reshape(1, -1)
        prediction, evaluation_hw = predict_fn(image)
        if tuple(evaluation_hw) == original_gt.shape:
            gt = original_gt
        else:
            gt = F.interpolate(torch.from_numpy(original_gt.astype(np.float32))[None, None],
                               size=tuple(evaluation_hw), mode="nearest")[0, 0].numpy() > 0
        row = {**case, **binary_metrics(prediction, gt),
               "original_height": image.shape[0], "original_width": image.shape[1],
               "original_gt_area": int(original_gt.sum())}
        if save_masks:
            from PIL import Image
            # Hash the identity rather than flattening paths, which can collide.
            key = hashlib.sha256(f"{case['dataset']}:{case['case_id']}".encode()).hexdigest()[:20]
            path = output_dir / "pred_masks" / f"{key}.png"
            path.parent.mkdir(parents=True, exist_ok=True)
            Image.fromarray(np.asarray(prediction, dtype=np.uint8) * 255).save(path)
            row["prediction_path"] = str(path)
        rows.append(row)
        vectors.append(vector)
        if index % 10 == 0 or index == len(cases):
            print(f"Evaluation: {index}/{len(cases)}")
    if not vectors:
        raise ValueError("No evaluation cases")
    return rows, torch.cat(vectors)


def normalize_scores(raw_scores):
    scores = np.asarray(raw_scores, dtype=np.float64).reshape(-1)
    if not len(scores) or not np.isfinite(scores).all():
        raise ValueError("OOD scores must be nonempty and finite")
    low, high = float(scores.min()), float(scores.max())
    return ((scores - low) / (high - low) if high > low else np.zeros_like(scores),
            {"min": low, "max": high})


def attach_ood_scores(rows, id_features, evaluation_features, k=50, normalize=True):
    """K-th neighbor distance; defaults match SEG_OOD_medSAM's KNN detector."""
    from pytorch_ood.detector import KNN
    for name, matrix in [("ID", id_features), ("evaluation", evaluation_features)]:
        if matrix.ndim != 2 or not matrix.shape[0] or not torch.isfinite(matrix).all():
            raise ValueError(f"{name} features must be a nonempty finite (N, C) matrix")
    if id_features.shape[1] != evaluation_features.shape[1]:
        raise ValueError("ID/evaluation feature dimensions differ; check the encoder and checkpoint")
    if len(rows) != len(evaluation_features):
        raise ValueError("Number of feature vectors differs from number of metric rows")
    if not 1 <= k <= len(id_features):
        raise ValueError(f"KNN_K must be between 1 and the ID reference count ({len(id_features)})")
    detector = KNN(encoder=None, k=k, normalize=normalize)
    detector.fit_features(id_features.detach().cpu().float().clone(),
                          torch.zeros(len(id_features), dtype=torch.long))
    raw = torch.as_tensor(detector.predict_features(evaluation_features.detach().cpu().float().clone())).cpu().numpy()
    normalized, scaling = normalize_scores(raw)
    for row, raw_score, score in zip(rows, raw, normalized):
        row.update(ood_score_raw=float(raw_score), ood_score=float(score))
    return scaling

## Segmentation model definitions

In [ ]:
class VersaMammoAdapter:
    """Reuse VersaMammo's segmentation architecture and train_seg_v2 preprocessing.

    Instantiate without pretrained downloads, then strictly restore the complete
    segmentation state dict. The deepest encoder7 map can provide OOD features.
    """
    def __init__(self, checkpoint, device, input_size=512, normalization="percentile",
                 percentile_low=1.0, percentile_high=99.0, threshold=0.5):
        from versamammo_train_seg_v2 import UNetEfficientNetB5, _to_chw_image
        from versamammo_evaluate_mamamia import load_state_dict
        if normalization not in {"percentile", "minmax", "legacy"}:
            raise ValueError("Unknown intensity normalization")
        if not 0 <= percentile_low < percentile_high <= 100 or not 0 <= threshold <= 1:
            raise ValueError("Invalid percentiles or mask threshold")
        if input_size < 64:
            raise ValueError("input_size must be at least 64 for encoder7 + pooling")
        self.device, self.size = torch.device(device), (input_size, input_size)
        self.normalization, self.low, self.high = normalization, percentile_low, percentile_high
        self.threshold, self.to_chw = threshold, _to_chw_image
        state = load_state_dict(Path(checkpoint), torch.device("cpu"))
        self.model = UNetEfficientNetB5(pretrained=False)
        # The pretrained B5 used by training may have a 1-class classifier;
        # from_name defaults to 1000. It is restored although U-Net doesn't use it.
        if "backbone._fc.weight" in state:
            out_features, in_features = state["backbone._fc.weight"].shape
            self.model.backbone._fc = torch.nn.Linear(in_features, out_features)
        self.model.load_state_dict(state, strict=True)
        self.model.to(self.device).eval()

    def preprocess(self, image):
        tensor = self.to_chw(image, Path("input"), self.normalization, self.low, self.high)
        tensor = F.interpolate(tensor[None], size=self.size, mode="bilinear", align_corners=False)
        # Matches myNormalize used by the evaluation DataLoader.
        return ((tensor - 0.5) / 0.5).to(self.device)

    @torch.inference_mode()
    def features(self, image):
        tensor = self.preprocess(image)
        for index in range(8):
            tensor = getattr(self.model, f"encoder{index}")(tensor)
        return tensor.mean(dim=(2, 3)).cpu()

    @torch.inference_mode()
    def predict(self, image):
        # UNetEfficientNetB5.forward already returns sigmoid probabilities.
        prediction = self.model(self.preprocess(image))[0, 0] >= self.threshold
        return prediction.cpu().numpy(), self.size

def load_detection_vit_builder(project_root):
    """Load Detection's image-to-token ViT without shadowing Segment imports.

    Use an isolated package name so Jupyter's cached VersaMammo modules and
    sys.path ordering cannot select Segment's token-input implementation.
    """
    import importlib
    import sys
    from types import ModuleType

    package_dir = (Path(project_root) / "VersaMammo" / "downstream" / "Detection" / "VersaMammo").resolve()
    module_file = package_dir / "models" / "image_encoder.py"
    if not module_file.is_file():
        raise FileNotFoundError(module_file)
    package_name = "_seg_ood_versamammo_detection"
    package = sys.modules.get(package_name)
    if package is None:
        package = ModuleType(package_name)
        package.__path__ = [str(package_dir)]
        package.__package__ = package_name
        sys.modules[package_name] = package
    elif list(package.__path__) != [str(package_dir)]:
        raise RuntimeError("The Detection ViT was loaded from another project root; restart the kernel")
    module = importlib.import_module(f"{package_name}.models.image_encoder")
    if Path(module.__file__).resolve() != module_file:
        raise RuntimeError(f"Unexpected ViT builder source: {module.__file__}")
    print("OOD ViT implementation:", module.vits.__file__)
    return module.build_vit_model


class VersaMammoViTFeatures:
    """768-dimensional ViT-B/14 features with the original extractor's preprocessing.

    Supply the exact encoder state used for the saved ID vectors. Strict loading
    prevents an EfficientNet checkpoint from silently leaving random ViT weights.
    """
    def __init__(self, checkpoint, device):
        build_vit_model = load_detection_vit_builder(PROJECT_ROOT)
        checkpoint = Path(checkpoint)
        if not checkpoint.is_file():
            raise FileNotFoundError(
                f"ViT encoder state not found: {checkpoint}. Set VIT_CHECKPOINT to the "
                "exact ViT weights used for the saved 768-dimensional ID features. "
                "The EfficientNet (Enb5) checkpoint does not contain those weights."
            )
        self.encoder, num_features = build_vit_model(arch="vit_base", pretrained=False)
        if num_features != 768:
            raise ValueError(f"Expected ViT-B features with 768 dimensions, got {num_features}")
        state = torch.load(checkpoint, map_location="cpu", weights_only=True)
        for wrapper in ("teacher", "model", "model_state_dict", "state_dict"):
            if wrapper in state and isinstance(state[wrapper], dict):
                state = state[wrapper]
                break
        expected = self.encoder.state_dict()
        cleaned = {}
        for name, value in state.items():
            for prefix in ("module.", "image_encoder.", "backbone."):
                name = name.removeprefix(prefix)
            if name in expected:
                cleaned[name] = value
        try:
            self.encoder.load_state_dict(cleaned, strict=True)
        except RuntimeError as exc:
            raise ValueError("The checkpoint does not contain a complete compatible ViT-B/14 "
                             "encoder state. Use the original feature encoder's weights.") from exc
        self.device = torch.device(device)
        self.encoder.to(self.device).eval()

    @torch.inference_mode()
    def features(self, image):
        image = as_2d(image).astype(np.float32)
        image = (image - image.min()) / max(float(image.max() - image.min()), 1e-8)
        tensor = torch.from_numpy(image)[None, None].repeat(1, 3, 1, 1)
        tensor = F.interpolate(tensor, size=(224, 224), mode="bilinear", align_corners=False)
        features = self.encoder(((tensor - 0.5) / 0.5).to(self.device))
        # Same pooling as pool_encoder_features in the extraction notebooks.
        if isinstance(features, (tuple, list)):
            features = features[0]
        if features.ndim == 4:
            features = features.mean(dim=(2, 3))
        elif features.ndim == 3:
            features = features.mean(dim=1)
        elif features.ndim != 2:
            features = features.flatten(start_dim=1)
        if features.shape != (1, 768):
            raise ValueError(f"Expected one 768-dimensional ViT feature vector, got {features.shape}")
        return features.cpu()

## Model loading and feature space

The segmentation architecture already returns sigmoid probabilities; the adapter thresholds
them once. It reuses `versamammo_train_seg_v2._to_chw_image` and the same resize and
normalization as the NPZ path in `versamammo_evaluate_mamamia.py`, without writing dataset caches.
Set preprocessing to the policy appropriate for your checkpoint before running.
For the PNG dataframe trainer, `legacy` matches uint8 `/255`; `minmax` matches its minmax mode.

The pipeline loads the existing 768-dimensional ID features by default and uses a
separate ViT encoder for evaluation features. The ViT uses image min-max normalization,
three channels, a 224 x 224 bilinear resize, mean/std 0.5, and spatial/token pooling.
Segmentation retains its own EfficientNet preprocessing. ViT weights load strictly;
matching dimensions alone is not proof that the ID and evaluation feature spaces match.

The OOD ViT is loaded explicitly from `downstream/Detection` under an isolated
package name. This avoids importing Segment's ViT, which expects precomputed tokens
and skips image patch embedding. EfficientNet segmentation uses the Segment code.


In [12]:
adapter = VersaMammoAdapter(
    SEGMENTATION_CHECKPOINT, DEVICE, SEGMENTATION_INPUT_SIZE,
    SEGMENTATION_NORMALIZATION, PERCENTILE_LOW, PERCENTILE_HIGH, MASK_THRESHOLD,
)
segmentation_signature = {
    "checkpoint": checkpoint_identity(SEGMENTATION_CHECKPOINT),
    "input_size": SEGMENTATION_INPUT_SIZE,
    "normalization": SEGMENTATION_NORMALIZATION,
    "percentiles": [PERCENTILE_LOW, PERCENTILE_HIGH],
    "mask_threshold": MASK_THRESHOLD,
}
vit_features = VersaMammoViTFeatures(VIT_CHECKPOINT, DEVICE)
feature_fn = vit_features.features
feature_signature = {
    "encoder": "VersaMammo ViT-B/14 spatial/token mean",
    "checkpoint": checkpoint_identity(VIT_CHECKPOINT),
    "feature_dimensions": 768,
    "input_size": 224,
    "normalization": "image minmax then mean=std=0.5",
}
DICE_RESOLUTION = f"resized {SEGMENTATION_INPUT_SIZE} x {SEGMENTATION_INPUT_SIZE}"
PLOT_TITLE = "EfficientNet-B5 segmentation Dice vs ViT OOD distance"

## Select evaluation images

In [13]:
evaluation_cases = []
for dataset_name, dataset_root in EVALUATION_ROOTS.items():
    cases = collect_paired_cases(dataset_root, dataset_name, MAX_EVALUATION_CASES_PER_DATASET)
    evaluation_cases.extend(cases)
    print(dataset_name, len(cases))

# No OOD ordering or filtering: all selected images are evaluated.
# For a different directory structure, replace evaluation_cases with records:
# {"dataset": "mamamia", "case_id": "unique_id", "image_path": "...", "mask_path": "..."}

mamamia 171
inbreast 108


## Extract or load the ID reference

In [14]:
if REEXTRACT_ID_FEATURES:
    id_cases = collect_zgt_cases(ZGT_DATA_ROOT, ZGT_METADATA_CSV, MAX_ID_CASES)
    id_features = extract_reference_features(
        id_cases, feature_fn, ID_FEATURES_PATH, feature_signature,
    )
else:
    id_features = load_reference_features(
        ID_FEATURES_PATH, feature_signature, allow_legacy=ALLOW_LEGACY_ID_FEATURES,
    )
print("ID reference features:", tuple(id_features.shape))
if id_features.ndim != 2 or id_features.shape[1] != 768:
    raise ValueError(f"Expected saved ID features shaped (N, 768), got {tuple(id_features.shape)}")
if not 1 <= KNN_K <= len(id_features):
    raise ValueError(f"Set KNN_K between 1 and {len(id_features)} before evaluating images.")

# Check the actual evaluation encoder output before segmenting the full datasets.
probe_image, _ = read_case(
    evaluation_cases[0], image_npz_key=IMAGE_NPZ_KEY, mask_npz_key=MASK_NPZ_KEY,
)
probe_features = feature_fn(probe_image).detach().cpu().reshape(1, -1)
print("Evaluation ViT feature shape:", tuple(probe_features.shape))
if id_features.shape[1] != probe_features.shape[1]:
    raise ValueError(
        f"ID features have {id_features.shape[1]} dimensions; current ViT "
        f"features have {probe_features.shape[1]}. Check feature_fn, loaded weights "
        "and pooling. Rerun the model-definition/setup cells if the kernel has stale state."
    )

/tmp/ipykernel_315/825652061.py:35: UserWarning: Using legacy ID features: verify the same encoder weights, fold and preprocessing
  warnings.warn("Using legacy ID features: verify the same encoder weights, fold and preprocessing")


ID reference features: (159, 768)


RuntimeError: Given normalized_shape=[768], expected input with shape [*, 768], but got input of size[1, 3, 224, 224]

## Extract evaluation features, segment, and score OOD distance

In [ ]:
rows, evaluation_features = evaluate_cases(
    evaluation_cases, feature_fn, adapter.predict, OUTPUT_ROOT,
    save_masks=SAVE_PRED_MASKS,
    image_npz_key=IMAGE_NPZ_KEY, mask_npz_key=MASK_NPZ_KEY,
)
score_scaling = attach_ood_scores(
    rows, id_features, evaluation_features, k=KNN_K, normalize=KNN_NORMALIZE_FEATURES,
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
torch.save({"features": evaluation_features, "signature": feature_signature,
            "cases": evaluation_cases}, OUTPUT_ROOT / "evaluation_features.pt")
print("Joint OOD scaling:", score_scaling)

## Correlation analysis

In [ ]:
def save_analysis(rows, output_dir, title, config):
    """Save paired metrics, score-sorted ranking, per-dataset correlations and plot."""
    import matplotlib.pyplot as plt
    from scipy.stats import pearsonr, spearmanr

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fields = sorted(set().union(*(row.keys() for row in rows)))
    for filename, records in [("per_image_metrics.csv", rows),
                              ("ranked_by_ood.csv", sorted(rows, key=lambda r: r["ood_score_raw"]))]:
        with (output_dir / filename).open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=fields)
            writer.writeheader()
            writer.writerows(records)
    fig, ax = plt.subplots(figsize=(8, 6))
    summary = {}
    dataset_colors = {"mamamia": "tab:blue", "inbreast": "tab:orange"}
    for dataset in sorted({r["dataset"] for r in rows}):
        records = [r for r in rows if r["dataset"] == dataset]
        x = np.asarray([r["ood_score"] for r in records])
        y = np.asarray([r["dice"] for r in records])
        valid = np.isfinite(x) & np.isfinite(y)
        x, y = x[valid], y[valid]
        stats = {"n": len(x), "mean_dice": float(y.mean()) if len(y) else None,
                 "mean_ood_raw": float(np.mean([r["ood_score_raw"] for r in records])),
                 "pearson_r": None, "pearson_p": None, "spearman_r": None, "spearman_p": None}
        scatter = ax.scatter(x, y, alpha=0.6, label=dataset, color=dataset_colors.get(dataset))
        if len(x) >= 2 and np.ptp(x) > 0 and np.ptp(y) > 0:
            r, p = pearsonr(x, y)
            rho, sp = spearmanr(x, y)
            stats.update(pearson_r=float(r), pearson_p=float(p) if np.isfinite(p) else None,
                         spearman_r=float(rho), spearman_p=float(sp) if np.isfinite(sp) else None)
            grid = np.linspace(x.min(), x.max(), 100)
            ax.plot(grid, np.polyval(np.polyfit(x, y, 1), grid),
                    color=scatter.get_facecolor()[0], label=f"{dataset}: r={r:.3f}")
        else:
            stats["correlation_note"] = "Requires at least two samples and nonconstant scores"
        summary[dataset] = stats
    ax.set(xlabel="OOD score (joint min-max normalization)", ylabel="Dice", title=title)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "dice_vs_ood.png", dpi=300)
    with (output_dir / "summary.json").open("w") as handle:
        json.dump({"datasets": summary, "config": config}, handle, indent=2, default=str, allow_nan=False)
    return fig, summary

In [ ]:
run_config = {
    "feature_signature": feature_signature,
    "segmentation_signature": segmentation_signature,
    "id_features_path": str(ID_FEATURES_PATH),
    "zgt_data_root": str(ZGT_DATA_ROOT), "zgt_metadata_csv": str(ZGT_METADATA_CSV),
    "reextracted_id_features": REEXTRACT_ID_FEATURES,
    "allow_legacy_id_features": ALLOW_LEGACY_ID_FEATURES,
    "id_reference_count": len(id_features),
    "evaluation_roots": {k: str(v) for k, v in EVALUATION_ROOTS.items()},
    "max_id_cases": MAX_ID_CASES,
    "max_evaluation_cases_per_dataset": MAX_EVALUATION_CASES_PER_DATASET,
    "image_npz_key": IMAGE_NPZ_KEY, "mask_npz_key": MASK_NPZ_KEY,
    "score_scaling": score_scaling,
    "dice_resolution": DICE_RESOLUTION,
    "empty_mask_dice": 1.0,
    "ood_detector": "pytorch_ood.detector.KNN",
    "knn_k": KNN_K, "knn_normalize_features": KNN_NORMALIZE_FEATURES,
}
figure, summary = save_analysis(rows, OUTPUT_ROOT, PLOT_TITLE, run_config)
print(json.dumps(summary, indent=2))
print("Results:", OUTPUT_ROOT)
figure

Each point is an image/slice with a paired OOD score and Dice. A negative correlation
supports an association between greater ID feature distance and worse segmentation.
The raw score is distance to the 50th nearest ID feature by default, after L2
normalizing both reference and evaluation vectors. Set `KNN_K` to change this.
The score is not an OOD probability. Joint min-max scaling uses both evaluation datasets;
retain raw scores when comparing different runs. Pearson and Spearman correlations are
computed **separately per dataset**, avoiding a pooled correlation driven by dataset differences.

Multiple slices from one MRI patient are not independent observations; the reported
p-values are descriptive image-level statistics. For patient-level inference, aggregate
by patient or use patient-level resampling. Foreground area/fraction is saved to help
inspect whether lesion size relates to either distance or Dice. Empty GT + empty prediction
has Dice 1; one empty mask has Dice 0.

Outputs: `per_image_metrics.csv`, `ranked_by_ood.csv` (ascending score, for inspection),
`summary.json`, `dice_vs_ood.png`, `evaluation_features.pt`, and optional `pred_masks/`.
Masks are stored at the evaluation resolution and referenced from the CSV.

Dataset identity is retained in every metric row and feature case record. MAMA-MIA
is blue and INbreast is orange, with separate correlations and trend lines. Joint
score normalization does not merge their identities or average their measurements.